In [ ]:
# ============================================================================
# Imports
# ============================================================================

from pathlib import Path

import pandas as pd

from hermes.data_catalog import get_dataset
from hermes.io import (download_file, save_dataframe)
from hermes.preprocessing import prepare_mayotte_workplace_employment
from hermes.integration import integrate_workplace_employment
from hermes.loaders import load_workplace_employment
from hermes.config import PREPARED_DIR

In [ ]:
# ============================================================================
# Get dataset
# ============================================================================

dataset = get_dataset(
    "mayotte_workplace_employment_raw"
)

print(dataset.download_path)
print(dataset.local_path)

In [ ]:
# ============================================================================
# Download dataset
# ============================================================================

archive_path = download_file(
    url=dataset.url,
    destination=dataset.download_path,
)

In [ ]:
# ============================================================================
# Inspect extracted workbooks
# ============================================================================

for path in sorted(
    archive_path.parent.rglob("*.xls")
):
    if "EMP" in path.name.upper():
        print(path)

In [ ]:
# ============================================================================
# Inspect EMP1 workbook
# ============================================================================

workplace_files = {
    path.stem: path
    for path in archive_path.parent.glob("BTX_TD_EMP*_2017.xls")
}

workplace_files

In [ ]:
for name, path in workplace_files.items():
    workbook = pd.ExcelFile(
        path,
        engine="xlrd",
    )

    print(name, workbook.sheet_names)

In [ ]:
for name, path in workplace_files.items():

    preview = pd.read_excel(
        path,
        sheet_name="COM",
        engine="xlrd",
        header=None,
        nrows=3,
    )

    print(name)
    print(preview.iloc[0, 0])
    print()

In [ ]:
# ============================================================================
# Inspect EMP1 commune table
# ============================================================================

emp1_path = workplace_files[
    "BTX_TD_EMP1_2017"
]

emp1_raw = pd.read_excel(
    emp1_path,
    sheet_name="COM",
    engine="xlrd",
    header=None,
)

emp1_raw.head(20)

In [ ]:
# ============================================================================
# Inspect EMP1 variables
# ============================================================================

emp1_variables = pd.read_excel(
    emp1_path,
    sheet_name="Liste des variables",
    engine="xlrd",
    header=None,
)

emp1_variables.head(50)

In [ ]:
# ============================================================================
# Load full data for France and Mayotte
# ============================================================================

mayotte_workplace_employment = (
    prepare_mayotte_workplace_employment(
        emp1_raw,
    )
)

# Check 17x6
print(mayotte_workplace_employment.shape)
mayotte_workplace_employment.head()

In [ ]:
# ============================================================================
# Integrate workplace employment data
# ============================================================================

workplace_employment = load_workplace_employment()

workplace_employment_integrated = integrate_workplace_employment(
    workplace_employment,
    mayotte_workplace_employment,
)

In [ ]:
# Sanity check
print(workplace_employment.shape)
print(mayotte_workplace_employment.shape)
print(workplace_employment_integrated.shape)

In [ ]:
workplace_employment_integrated[
    workplace_employment_integrated["insee_code"] == "97602"
]

In [ ]:
# ============================================================================
# Save integrated workplace employment data
# ============================================================================

save_dataframe(
    workplace_employment_integrated,
    PREPARED_DIR / "workplace_employment.parquet",
)
